In [8]:
import importlib
import sys

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [9]:
# Load the data
file_path_train = '../../../../../../data/BPIC20_DD/tensor_data/normal/bpic20_dd_all_5_train.pkl'
bpic20_dd_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/BPIC20_DD/tensor_data/normal/bpic20_dd_all_5_val.pkl'
bpic20_dd_val_dataset = torch.load(file_path_val, weights_only=False)

In [10]:
# BPIC20_DD dataset dynamic categories, features:
bpic20_dd_all_categories = bpic20_dd_train_dataset.all_categories

bpic20_dd_all_categories_cat = bpic20_dd_all_categories[0]
bpic20_dd_all_categories_num = bpic20_dd_all_categories[1]
for i, cat in enumerate(bpic20_dd_all_categories_cat):
     print(f"BPIC20_DD dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(bpic20_dd_all_categories_num):
     print(f"BPIC20_DD dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of numerical: {num[1]}")
print('\n')
     
# BPIC20_DD dataset static categories, features:
bpic20_dd_all_stat_categories = bpic20_dd_train_dataset.all_static_categories

bpic20_dd_all_stat_categories_cat = bpic20_dd_all_stat_categories[0]
bpic20_dd_all_stat_categories_num = bpic20_dd_all_stat_categories[1]
for i, cat in enumerate(bpic20_dd_all_stat_categories_cat):
     print(f"BPIC20_DD static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(bpic20_dd_all_stat_categories_num):
     print(f"BPIC20_DD static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of numerical: {num[1]}")

BPIC20_DD dynamic categorical feature: concept:name, Index position in categorical data list: 0
BPIC20_DD amount of category labels: 10
BPIC20_DD dynamic categorical feature: org:resource, Index position in categorical data list: 1
BPIC20_DD amount of category labels: 4
BPIC20_DD dynamic categorical feature: org:role, Index position in categorical data list: 2
BPIC20_DD amount of category labels: 9


BPIC20_DD dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: day_in_week, Index position in categorical data list: 2
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: seconds_in_day, Index position in categorical data list: 3
BPIC20_DD amount of numerical: 1


BPIC20_DD static categorical feature: case:BudgetNumber, Index position in

In [11]:
# Create lists with name of Encoder features (input) and decoder features (input & output)

# Encoder features (fixed): use only requested features
enc_feat_cat = ['concept:name', 'org:resource']
enc_feat_num = ['case_elapsed_time']
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Decoder features (unused by C-LSTM training cell, kept for consistency)
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

Input features encoder:  [['concept:name', 'org:resource'], ['case_elapsed_time']]
Features decoder:  [['concept:name'], []]


In [12]:
import suffix_pred.models.C_LSTM
importlib.reload(suffix_pred.models.C_LSTM)
from suffix_pred.models.C_LSTM import FullShared_Join_LSTM

# Size hidden layer
hidden_size = 50

# Number of LSTM cells
num_layers = 1

# STANDARD: One numerical output to predict
input_size = 1

# C-LSTM uses dynamic prefix features only
model_feat = enc_feat

# Output size: activity classes only
activity_feature_name = 'concept:name'
activity_feature_index = next(i for i, cat in enumerate(bpic20_dd_all_categories_cat) if cat[0] == activity_feature_name)
output_size_act = bpic20_dd_all_categories_cat[activity_feature_index][1]

# C-LSTM model initialization
model = FullShared_Join_LSTM(data_set_categories=bpic20_dd_all_categories,
                             hidden_size=hidden_size,
                             num_layers=num_layers,
                             model_feat=model_feat,
                             input_size=input_size,
                             output_size_act=output_size_act)

Data set categories:  ([('concept:name', 10, {'Declaration APPROVED': 1, 'Declaration FINAL_APPROVED': 2, 'Declaration FOR_APPROVAL': 3, 'Declaration REJECTED': 4, 'Declaration SAVED': 5, 'Declaration SUBMITTED': 6, 'EOS': 7, 'Payment Handled': 8, 'Request Payment': 9}), ('org:resource', 4, {'EOS': 1, 'STAFF MEMBER': 2, 'SYSTEM': 3}), ('org:role', 9, {'ADMINISTRATION': 1, 'BUDGET OWNER': 2, 'EMPLOYEE': 3, 'EOS': 4, 'MISSING': 5, 'PRE_APPROVER': 6, 'SUPERVISOR': 7, 'UNDEFINED': 8})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {})])
Model input features:  [['concept:name', 'org:resource'], ['case_elapsed_time']]


Embeddings:  ModuleList(
  (0): Embedding(10, 6)
  (1): Embedding(4, 3)
)
Total embedding feature size:  9
Input feature size:  10
Cells hidden size:  50
Number of LSTM layer:  1


/home/PSPLab/.local/share/virtualenvs/decision_aware_augmentation_for_PPM-0DzgvVpC/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [13]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [14]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import CTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 0.001
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)


# Keep consistent across all models
# Epochs / Batch size
num_epochs = 100
batch_size = 128

# Shuffle data
shuffle = True

optimize_values = {"optimizer": optimizer,
                   "scheduler": scheduler,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle}

# Activity feature index and EOS id for activity-only next-event prediction
activity_feature_name = 'concept:name'
concept_name_id = next(i for i, cat in enumerate(bpic20_dd_all_categories_cat) if cat[0] == activity_feature_name)
activity_label_to_id = bpic20_dd_all_categories_cat[concept_name_id][2]
eos_id = activity_label_to_id.get('EOS')

model_output_path = "../../../../../../models/BPIC20_DD/clean/BPIC20_DD_C_LSTM_v1_clean.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = CTraining(device=device,
                    model=model,
                    data_train=bpic20_dd_train_dataset,
                    data_val=bpic20_dd_val_dataset,
                    optimize_values=optimize_values,
                    concept_name_id=concept_name_id,
                    eos_id=eos_id,
                    loss_obj=loss_obj,
                    save_model_n_th_epoch=1,
                    saving_path=model_output_path)

# Train the model (activity sequence modeling via next-activity training)
trainer.train()

Device: cuda
Device:  cuda
Optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7fb746bc4620>
Epochs:  100
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.3047
Validation: Avg Validation Loss: 0.1262
Validation Loss for Scheduler: 0.1262
saving model
Epoch [2/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.0901
Validation: Avg Validation Loss: 0.0992
Validation Loss for Scheduler: 0.0992
saving model
Epoch [3/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.0783
Validation: Avg Validation Loss: 0.0948
Validation Loss for Scheduler: 0.0948
saving model
Epoch [4/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.0740
Validation: Avg Validation Loss: 0.0895
Validation Loss for Scheduler: 0.0895
saving model
Epoch [5/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.0717
Validation: Avg Validation Loss: 0.0895
Validation Loss for Scheduler: 0.0895
saving model
Epoch [6/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.0708
Validation: Avg Validation Loss: 0.0874
Validat

([0.3046982606600334,
  0.09007928456514035,
  0.07825265503780533,
  0.07397586335217132,
  0.07170931072333724,
  0.07077287856626523,
  0.06917227550189883,
  0.06751922527161826,
  0.06679022136162188,
  0.0677884549054825,
  0.06565004701540361,
  0.06572333484736372,
  0.06502268485175705,
  0.06569603855522084,
  0.06445100779297522,
  0.06378460552106134,
  0.0630991319094043,
  0.06240599192441636,
  0.06384088664124184,
  0.06508933981671242,
  0.06206424249733918,
  0.06211606227993433,
  0.061607765627985187,
  0.062075244030896544,
  0.06090747096036007,
  0.059054011040333115,
  0.06012980643856458,
  0.060133491931403035,
  0.05967127517336665,
  0.05971127615154921,
  0.05904556423288167,
  0.058797707538629916,
  0.057686299372801904,
  0.05742623049782861,
  0.057340968985228835,
  0.05696099034908272,
  0.05560934458742038,
  0.055145238520687916,
  0.054752636949721566,
  0.054569901214604484,
  0.05460625938675285,
  0.05388652702313012,
  0.05430441171268345,
  0.